In [1]:
import panel as pn
import holoviews as hv
import pandas as pd
from corePlanner import get_targets, get_target_airmass, get_sun_rise_set, parse_ephemeris, determine_moon_phase
import PES_secrets

pn.extension('tabulator')
# Ensure Panel is initialized
pn.extension()


In [2]:
# this cell loads the targets for the night
datestr =pd.Timestamp.now().strftime("%Y-%m-%d")
targets_df = get_targets()  # Fetch the targets DataFrame

# while testing, set the targets_df to a small subset
#targets_df = targets_df.head(5)
# initialize the ephemeris column null string
targets_df['ephemeris'] = None

In [3]:
# this cell sets up some other text boxes for details of ther night
# get the sunset and sunrise times
sunset, sunrise = get_sun_rise_set(datestr)
# convert the sunset from a timestAMP IN SECONDS TO timezone of utc to local time
sunset = pd.to_datetime(sunset, unit='s').tz_localize('UTC').tz_convert(PES_secrets.obszone)
# convert the sunrise from a timezone of utc to local time
sunrise = pd.to_datetime(sunrise, unit='s').tz_localize('UTC').tz_convert(PES_secrets.obszone)
# create a str pane for the sunset time where there is a title centered and below that the value
sunset_pane = pn.pane.Markdown(f"Sunset\n {sunset.strftime('%H:%M:%S')}", width=120)
# create a str pane for the sunrise time
sunrise_pane = pn.pane.Markdown(f"Sunrise\n {sunrise.strftime('%H:%M:%S')}", width=120)
# create a str pane for the moon phase
moon_phase = determine_moon_phase(datestr)
# create a str pane for the moon phase phase to 0 dp
moon_phase_pane = pn.pane.Markdown(f"Moon Phase\n {moon_phase:.1f}", width=120)

In [4]:

targets_df = get_targets()  # Fetch the targets DataFrame

# while testing, set the targets_df to a small subset
#targets_df = targets_df.head(5)
# initialize the ephemeris column null string
targets_df['ephemeris'] = None


In [5]:
# now enrich the targets_df with event information
# create a widget to show the progress of the ephemeris parsing
# create a progress bar as a pane in the layout

# set the progress bar to be 0 to len(targets_df)

# create the progress bar
ephem_progress_value = 0
ephem_progress = pn.widgets.Progress(
    value=ephem_progress_value,
    active=True,
    max=len(targets_df),
    width=80,
    bar_color='primary',
    height=20
)
# set the title of the progress bar
ephem_progress.title = "Parsing Ephemeris"

# update the progress bar
def update_progress_bar(value):
    ephem_progress.value = value

# create a button to start the ephemeris parsing
process_ephem_button = pn.widgets.Button(
    name='Process Ephemeris',
    button_type='primary',
    width=120,
    height=50
)

# Function to process ephemeris for each target
def process_ephemeris(event):
    # set the process button to disabled
    process_ephem_button.disabled = True
    # Iterate over each target and parse ephemeris
    for index, row in targets_df.iterrows():
        # print a progress message
        print(f"Processing target {index + 1} of {len(targets_df)}: {row['star_name']}")
        # check if the ephemeris is empty
        if row['other_info'] is None:
            continue
        # add the ephemeris to the targetdf
        targets_df.at[index, 'ephemeris'] = parse_ephemeris(row['other_info'])
    # update the progress bar
        update_progress_bar(index + 1)
    # when the ephemeris is done, set the button to enabled
    process_ephem_button.disabled = False
    # update the targets table
    targets_table.value = targets_df
# reset the index to start from 0
targets_df.reset_index(drop=True, inplace=True)


# set the on_click event of the button to start the ephemeris parsing
process_ephem_button.on_click(process_ephemeris)

# Create a Tabulator widget for interactive row selection
# only show the columns that are needed
target_table_df = targets_df[['star_name', 'ra', 'dec', 'ephemeris', 'other_info']]
# convert ra from degrees to hh:mm:ss
target_table_df['ra'] = targets_df['ra'].apply(lambda x: f"{int(x // 15):02}:{int((x % 15) * 4):02}:{int(((x % 15) * 4 % 1) * 60):02}")
# convert dec from degrees to dd:mm:ss
target_table_df['dec'] = targets_df['dec'].apply(lambda x: f"{int(x):02}:{int(abs(x) % 1 * 60):02}:{int((abs(x) % 1 * 60 % 1) * 60):02}")
targets_table = pn.widgets.Tabulator(
    target_table_df,
    selectable=1,  # Allow single row selection
    width=800,
    height=400
)

# set the title of targets_table
targets_table.title = "Targets for " + pd.Timestamp.now().strftime("%Y-%m-%d")


# Function to handle row selection
def on_row_select(event):
    selected_row = targets_table.selection
    if selected_row:
        selected_data = targets_df.iloc[selected_row[0]]  # Get the selected row data
        print("Selected Row Data:", selected_data)  # Replace with desired action
        # Update the display with selected row data

# Attach the row selection event to the Tabulator widget
targets_table.param.watch(on_row_select, 'selection')

# create a pane to display the selected row data
selected_row_pane = pn.pane.Str("No row selected", width=800)
# Update the pane with selected row data
def update_selected_row_pane(event):
    selected_row = targets_table.selection
    if selected_row:
        selected_data = targets_df.iloc[selected_row[0]]
        selected_row_pane.object = str(selected_data)
    else:
        selected_row_pane.object = "No row selected"
# Attach the update function to the Tabulator widget
targets_table.param.watch(update_selected_row_pane, 'selection')

# here is the function to generate the airmass graph
def generate_airmass_graph(selected_data):
    timezone = PES_secrets.obszone
    times, airmass = get_target_airmass(selected_data)
    print(f"Times: {times}")
    print(f"Airmass: {airmass}")
    # Filter out invalid times and airmass values
    valid_data = [(t, a) for t, a in zip(times, airmass) if 1 <= a <= 10]
    if not valid_data:
        return hv.Text(0.5, 0.5, "No valid data for airmass graph").opts(
            width=800, height=400
        )
    filtered_times, filtered_airmass = zip(*valid_data)
    
    # Convert times to timezone-aware datetime objects
    filtered_times = pd.to_datetime(filtered_times, unit='s', utc=True).tz_convert(timezone)
    # Create a Holoviews plot of airmass vs times
    # so... the scatter object uses tz.naive so the tz must be removed
    airmass_pts = hv.Scatter((filtered_times, filtered_airmass), label=selected_data['star_name']).opts(
        title="Airmass vs Time",
        xlabel="Time",
        ylabel="Airmass",
        width=800,
        height=400,    
        invert_yaxis=True,
        ylim=(1, 3),
        xlim=(sunset.tz_localize(None), sunrise.tz_localize(None)),
        size=5,
        tools=['hover']
    )   
    airmass_curve = hv.Curve((filtered_times, filtered_airmass), label=selected_data['star_name']).opts(
         line_width=2
    )
    airmass_graph = airmass_pts * airmass_curve
    return airmass_graph
 

    # # # Add vertical lines for ephemeris times
    # # ephemeris = selected_data.get('ephemeris', None)
    # # if ephemeris is not None:
    # #     ephemeris_times = [pd.to_datetime(ephem, utc=True).tz_convert(timezone) for ephem in ephemeris]
    # #     for ephem_time in ephemeris_times:
    # #         airmass_graph *= hv.VLine(ephem_time).opts(
    # #             line_color='red',
    # #             line_width=2,
    # #             line_dash='dashed'
    # #         )
    # return airmass_graph
 
    # # set the title of the graph
    # airmass_graph.opts(
    #     title="Airmass vs Time for " + selected_data['star_name'],
    #     xlabel="Time",
    #     ylabel="Airmass")
    # return airmass_graph

# # Create a pane to display the airmass graph
airmass_pane = pn.pane.HoloViews(
    generate_airmass_graph(targets_df.iloc[0]),  # Initial graph with the first target
    width=800,
    height=400
)

# Function to update the airmass graph based on selected row
def update_airmass_graph(event):
    selected_row = targets_table.selection
    if selected_row:
        selected_data = targets_df.iloc[selected_row[0]]
        # call generate_airmass_graph function to create the graph
        airmass_graph = generate_airmass_graph(selected_data) 
        # Here you would generate the airmass graph based on selected_data
        # For demonstration, we'll just update the pane with a placeholder message
        airmass_pane.object = airmass_graph
# Attach the update function to the Tabulator widget
targets_table.param.watch(update_airmass_graph, 'selection')
# Create a layout for the airmass graph
airmass_layout = pn.Column(
    airmass_pane
)



/tmp/ipykernel_1600477/1761584198.py:62: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  target_table_df['ra'] = targets_df['ra'].apply(lambda x: f"{int(x // 15):02}:{int((x % 15) * 4):02}:{int(((x % 15) * 4 % 1) * 60):02}")
/tmp/ipykernel_1600477/1761584198.py:64: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  target_table_df['dec'] = targets_df['dec'].apply(lambda x: f"{int(x):02}:{int(abs(x) % 1 * 60):02}:{int((abs(x) % 1 * 60 % 1) * 60):02}")


Times: [1746170529.802023, 1746174129.802023, 1746177729.802023, 1746181329.802023, 1746184929.802023, 1746188529.802023, 1746192129.802023, 1746195729.802023, 1746199329.802023, 1746202929.802023, 1746206529.802023, 1746210129.802023, 1746213729.802023, 1746217329.802023]
Airmass: [1.7694858736840853, 1.9833396369611735, 2.23894367261403, 2.522721799727965, 2.8017728557555186, 3.02203173509227, 3.1236453656783882, 3.073508008082686, 2.8884001075996055, 2.622854801364399, 2.335826231447305, 2.0680278253846116, 1.8389767730377702, 1.6537899565219818]


In [ ]:

# create a Panel layout
layout = pn.Column(
    pn.pane.Markdown("## AAVSO Targets for " + pd.Timestamp.now().strftime("%Y-%m-%d")),
    pn.Row(
        pn.Column(
            process_ephem_button,
            ephem_progress,
            sunset_pane,
            sunrise_pane,
            moon_phase_pane
    ),
        
        targets_table,
        airmass_layout
    ),
    pn.pane.Markdown("### Selected Row Data"),
    selected_row_pane)
# Display the layout in a Jupyter notebook
layout.servable()
# Alternatively, if you want to run this as a standalone script,
# you can use the following line to serve the panel:
pn.serve(layout, show=True)


Launching server at http://localhost:45925


/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:215: UserWarning: Discarding nonzero nanoseconds in conversion.
  return (cmin.to_pydatetime().replace(tzinfo=None),
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:216: UserWarning: Discarding nonzero nanoseconds in conversion.
  cmax.to_pydatetime().replace(tzinfo=None))
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:215: UserWarning: Discarding nonzero nanoseconds in conversion.
  return (cmin.to_pydatetime().replace(tzinfo=None),
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:216: UserWarning: Discarding nonzero nanoseconds in conversion.
  cmax.to_pydatetime().replace(tzinfo=None))
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:215: UserWarning: Discarding nonz

Selected Row Data: star_name                                                         VY Ret
ra                                                              51.81983
dec                                                            -61.25914
constellation                                                        Ret
var_type                                                              EA
min_mag                                                             8.47
min_mag_band                                                           V
max_mag                                                             7.89
max_mag_band                                                           V
period                                                          14.21605
obs_cadence                                                     1.421605
obs_mode                                                             All
obs_section                                        [Eclipsing Variables]
filter                          

/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:215: UserWarning: Discarding nonzero nanoseconds in conversion.
  return (cmin.to_pydatetime().replace(tzinfo=None),
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:216: UserWarning: Discarding nonzero nanoseconds in conversion.
  cmax.to_pydatetime().replace(tzinfo=None))
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:215: UserWarning: Discarding nonzero nanoseconds in conversion.
  return (cmin.to_pydatetime().replace(tzinfo=None),
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:216: UserWarning: Discarding nonzero nanoseconds in conversion.
  cmax.to_pydatetime().replace(tzinfo=None))
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:215: UserWarning: Discarding nonz

Selected Row Data: star_name                                                         AW Col
ra                                                                91.297
dec                                                            -32.73092
constellation                                                        Col
var_type                                                              EA
min_mag                                                             8.63
min_mag_band                                                           V
max_mag                                                              8.0
max_mag_band                                                           V
period                                                           10.3175
obs_cadence                                                       1.0316
obs_mode                                                             All
obs_section                                        [Eclipsing Variables]
filter                          

...

/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:215: UserWarning: Discarding nonzero nanoseconds in conversion.
  return (cmin.to_pydatetime().replace(tzinfo=None),
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:216: UserWarning: Discarding nonzero nanoseconds in conversion.
  cmax.to_pydatetime().replace(tzinfo=None))
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:215: UserWarning: Discarding nonzero nanoseconds in conversion.
  return (cmin.to_pydatetime().replace(tzinfo=None),
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:216: UserWarning: Discarding nonzero nanoseconds in conversion.
  cmax.to_pydatetime().replace(tzinfo=None))
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:215: UserWarning: Discarding nonz

Selected Row Data: star_name                                 V1208 Sco
ra                                        253.58875
dec                                       -41.86158
constellation                                   Sco
var_type                                         EA
min_mag                                         NaN
min_mag_band                                      V
max_mag                                        9.67
max_mag_band                                      V
period                                          NaN
obs_cadence                                     3.0
obs_mode                                        All
obs_section                   [Eclipsing Variables]
filter                                          All
other_info                                     None
priority                                       None
last_data_point                                 NaN
observability_times    [[TARGET_RISES, 1746183633]]
solar_conjunction                            

/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:215: UserWarning: Discarding nonzero nanoseconds in conversion.
  return (cmin.to_pydatetime().replace(tzinfo=None),
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:216: UserWarning: Discarding nonzero nanoseconds in conversion.
  cmax.to_pydatetime().replace(tzinfo=None))
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:215: UserWarning: Discarding nonzero nanoseconds in conversion.
  return (cmin.to_pydatetime().replace(tzinfo=None),
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:216: UserWarning: Discarding nonzero nanoseconds in conversion.
  cmax.to_pydatetime().replace(tzinfo=None))
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:215: UserWarning: Discarding nonz

Selected Row Data: star_name                                                      V1216 Cen
ra                                                             168.64779
dec                                                            -55.50233
constellation                                                        Cen
var_type                                                              EA
min_mag                                                            11.47
min_mag_band                                                           V
max_mag                                                            11.32
max_mag_band                                                           V
period                                                           3.80399
obs_cadence                                                     0.380399
obs_mode                                                             All
obs_section                                        [Eclipsing Variables]
filter                          

/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:215: UserWarning: Discarding nonzero nanoseconds in conversion.
  return (cmin.to_pydatetime().replace(tzinfo=None),
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:216: UserWarning: Discarding nonzero nanoseconds in conversion.
  cmax.to_pydatetime().replace(tzinfo=None))
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:215: UserWarning: Discarding nonzero nanoseconds in conversion.
  return (cmin.to_pydatetime().replace(tzinfo=None),
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:216: UserWarning: Discarding nonzero nanoseconds in conversion.
  cmax.to_pydatetime().replace(tzinfo=None))
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:215: UserWarning: Discarding nonz